In [2]:
# ============================================
# Instalar PySpark (necessário no Colab)
# ============================================
!pip -q install pyspark

# ============================================
# Importar bibliotecas
# ============================================
import requests
from pyspark.sql import SparkSession

# ============================================
# Inicializar Spark
# ============================================
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Case Nexa Credito - SQL com Spark")
    .getOrCreate()
)

# ============================================
# URL base do bucket público
# ============================================
BASE_URL = "https://public-data-for-technical-interview.s3.amazonaws.com/dinamica_estag_analytics"

files = [
    "clientes.csv",
    "propostas_credito.csv",
    "contratos.csv",
    "parcelas.csv",
    "pagamentos.csv"
]

# ============================================
# Download dos arquivos
# ============================================
for file_name in files:
    url = f"{BASE_URL}/{file_name}"
    response = requests.get(url)
    response.raise_for_status()

    with open(file_name, "wb") as f:
        f.write(response.content)

    print(f"{file_name} baixado com sucesso.")

# ============================================
# Leitura dos CSVs com Spark
# ============================================
clientes = spark.read.csv(
    "clientes.csv",
    header=True,
    inferSchema=True
)

propostas = spark.read.csv(
    "propostas_credito.csv",
    header=True,
    inferSchema=True
)

contratos = spark.read.csv(
    "contratos.csv",
    header=True,
    inferSchema=True
)

parcelas = spark.read.csv(
    "parcelas.csv",
    header=True,
    inferSchema=True
)

pagamentos = spark.read.csv(
    "pagamentos.csv",
    header=True,
    inferSchema=True
)

# ============================================
# Criar views temporárias para SQL
# ============================================
clientes.createOrReplaceTempView("clientes")
propostas.createOrReplaceTempView("propostas_credito")
contratos.createOrReplaceTempView("contratos")
parcelas.createOrReplaceTempView("parcelas")
pagamentos.createOrReplaceTempView("pagamentos")

# ============================================
# Validação
# ============================================
print("Views criadas com sucesso:")
print("- clientes")
print("- propostas_credito")
print("- contratos")
print("- parcelas")
print("- pagamentos")

print("\nContagem de registros:")

spark.sql("SELECT COUNT(*) AS total FROM clientes").show()
spark.sql("SELECT COUNT(*) AS total FROM propostas_credito").show()
spark.sql("SELECT COUNT(*) AS total FROM contratos").show()
spark.sql("SELECT COUNT(*) AS total FROM parcelas").show()
spark.sql("SELECT COUNT(*) AS total FROM pagamentos").show()

clientes.csv baixado com sucesso.
propostas_credito.csv baixado com sucesso.
contratos.csv baixado com sucesso.
parcelas.csv baixado com sucesso.
pagamentos.csv baixado com sucesso.
Views criadas com sucesso:
- clientes
- propostas_credito
- contratos
- parcelas
- pagamentos

Contagem de registros:
+-----+
|total|
+-----+
|  502|
+-----+

+-----+
|total|
+-----+
|  903|
+-----+

+-----+
|total|
+-----+
|  486|
+-----+

+-----+
|total|
+-----+
| 4389|
+-----+

+-----+
|total|
+-----+
| 3326|
+-----+



In [6]:
# ============================================================
# ONE BIG TABLE - NEXA CRÉDITO
# Grão final: 1 linha por PROPOSTA
#
# Fluxo:
# Cliente -> Proposta -> Contrato -> Parcela -> Pagamento
#
# Estratégia:
# - tratar tipos
# - agregar pagamentos por parcela
# - agregar parcelas por contrato
# - agregar contratos por proposta
# - juntar tudo somente no final
#
# Isso evita FANOUT / multiplicação indevida de valores.
# ============================================================

resultado_final = spark.sql("""

WITH

-- ============================================================
-- 1. CLIENTES
-- ============================================================

clientes_tratados AS (

    SELECT
        CAST(id_cliente AS BIGINT) AS id_cliente,

        tipo_cliente,

        -- Documento não deve ser tratado como medida numérica
        CAST(
            CAST(documento AS DECIMAL(38,0))
            AS STRING
        ) AS documento,

        COALESCE(
            TO_DATE(CAST(data_cadastro AS STRING), 'yyyy-MM-dd'),
            TO_DATE(CAST(data_cadastro AS STRING), 'dd/MM/yyyy')
        ) AS data_cadastro,

        cidade,
        uf,
        segmento,

        CAST(score_credito AS INT) AS score_credito,

        CAST(renda_mensal AS DECIMAL(18,2)) AS renda_mensal

    FROM clientes
),


-- ============================================================
-- 2. PROPOSTAS - LIMPEZA DO VALOR STRING
-- ============================================================

propostas_raw AS (

    SELECT
        *,

        REGEXP_REPLACE(
            TRIM(valor_solicitado),
            '[^0-9,.-]',
            ''
        ) AS valor_solicitado_txt

    FROM propostas_credito
),


propostas_tratadas AS (

    SELECT
        CAST(id_proposta AS BIGINT) AS id_proposta,

        COALESCE(
            TO_DATE(CAST(data_proposta AS STRING), 'yyyy-MM-dd'),
            TO_DATE(CAST(data_proposta AS STRING), 'dd/MM/yyyy')
        ) AS data_proposta,

        CAST(id_cliente AS BIGINT) AS id_cliente,

        canal_origem,
        produto_credito,

        -- Trata tanto:
        -- 23526,40
        -- quanto
        -- 23526.40
        CASE

            WHEN valor_solicitado_txt IS NULL
                 OR valor_solicitado_txt = ''
            THEN NULL

            WHEN INSTR(valor_solicitado_txt, ',') > 0
            THEN CAST(
                REPLACE(
                    REPLACE(
                        valor_solicitado_txt,
                        '.',
                        ''
                    ),
                    ',',
                    '.'
                )
                AS DECIMAL(18,2)
            )

            ELSE CAST(
                valor_solicitado_txt
                AS DECIMAL(18,2)
            )

        END AS valor_solicitado,

        CAST(prazo_meses AS INT) AS prazo_meses,

        CAST(
            taxa_mensal_solicitada
            AS DECIMAL(18,6)
        ) AS taxa_mensal_solicitada,

        status_proposta,
        motivo_recusa

    FROM propostas_raw
),


-- ============================================================
-- 3. CONTRATOS
-- ============================================================

contratos_tratados AS (

    SELECT
        CAST(id_contrato AS BIGINT) AS id_contrato,
        CAST(id_proposta AS BIGINT) AS id_proposta,

        COALESCE(
            TO_DATE(CAST(data_contratacao AS STRING), 'yyyy-MM-dd'),
            TO_DATE(CAST(data_contratacao AS STRING), 'dd/MM/yyyy')
        ) AS data_contratacao,

        CAST(valor_liberado AS DECIMAL(18,2))
            AS valor_liberado,

        CAST(taxa_mensal_final AS DECIMAL(18,6))
            AS taxa_mensal_final,

        CAST(prazo_final_meses AS INT)
            AS prazo_final_meses,

        status_contrato,
        canal_originacao,
        produto_credito_final

    FROM contratos
),


-- ============================================================
-- 4. PARCELAS
-- ============================================================

parcelas_tratadas AS (

    SELECT
        CAST(id_parcela AS BIGINT) AS id_parcela,
        CAST(id_contrato AS BIGINT) AS id_contrato,

        CAST(num_parcela AS INT) AS num_parcela,

        COALESCE(
            TO_DATE(CAST(data_vencimento AS STRING), 'yyyy-MM-dd'),
            TO_DATE(CAST(data_vencimento AS STRING), 'dd/MM/yyyy')
        ) AS data_vencimento,

        CAST(valor_parcela AS DECIMAL(18,2))
            AS valor_parcela,

        CAST(valor_principal AS DECIMAL(18,2))
            AS valor_principal,

        CAST(valor_juros AS DECIMAL(18,2))
            AS valor_juros,

        status_parcela

    FROM parcelas
),


-- ============================================================
-- 5. PAGAMENTOS - LIMPEZA DO VALOR STRING
-- ============================================================

pagamentos_raw AS (

    SELECT
        *,

        REGEXP_REPLACE(
            TRIM(valor_pago),
            '[^0-9,.-]',
            ''
        ) AS valor_pago_txt

    FROM pagamentos
),


pagamentos_tratados AS (

    SELECT
        CAST(id_pagamento AS BIGINT) AS id_pagamento,
        CAST(id_parcela AS BIGINT) AS id_parcela,

        COALESCE(
            TO_DATE(CAST(data_pagamento AS STRING), 'yyyy-MM-dd'),
            TO_DATE(CAST(data_pagamento AS STRING), 'dd/MM/yyyy')
        ) AS data_pagamento,

        CASE

            WHEN valor_pago_txt IS NULL
                 OR valor_pago_txt = ''
            THEN NULL

            WHEN INSTR(valor_pago_txt, ',') > 0
            THEN CAST(
                REPLACE(
                    REPLACE(
                        valor_pago_txt,
                        '.',
                        ''
                    ),
                    ',',
                    '.'
                )
                AS DECIMAL(18,2)
            )

            ELSE CAST(
                valor_pago_txt
                AS DECIMAL(18,2)
            )

        END AS valor_pago,

        CAST(valor_multa AS DECIMAL(18,2))
            AS valor_multa,

        CAST(valor_juros_mora AS DECIMAL(18,2))
            AS valor_juros_mora,

        CAST(valor_desconto AS DECIMAL(18,2))
            AS valor_desconto

    FROM pagamentos_raw
),


-- ============================================================
-- 6. AGREGA PAGAMENTOS
--    Grão: 1 linha por PARCELA
-- ============================================================

pagamentos_por_parcela AS (

    SELECT
        id_parcela,

        COUNT(DISTINCT id_pagamento)
            AS qtd_pagamentos,

        MIN(data_pagamento)
            AS data_primeiro_pagamento,

        MAX(data_pagamento)
            AS data_ultimo_pagamento,

        SUM(COALESCE(valor_pago, 0))
            AS valor_pago_total,

        SUM(COALESCE(valor_multa, 0))
            AS valor_multa_total,

        SUM(COALESCE(valor_juros_mora, 0))
            AS valor_juros_mora_total,

        SUM(COALESCE(valor_desconto, 0))
            AS valor_desconto_total

    FROM pagamentos_tratados

    GROUP BY
        id_parcela
),


-- ============================================================
-- 7. PARCELAS + PAGAMENTOS
--    Continua: 1 linha por PARCELA
-- ============================================================

parcelas_enriquecidas AS (

    SELECT
        p.id_parcela,
        p.id_contrato,
        p.num_parcela,
        p.data_vencimento,
        p.valor_parcela,
        p.valor_principal,
        p.valor_juros,
        p.status_parcela,

        COALESCE(pg.qtd_pagamentos, 0)
            AS qtd_pagamentos,

        pg.data_primeiro_pagamento,
        pg.data_ultimo_pagamento,

        COALESCE(pg.valor_pago_total, 0)
            AS valor_pago_total,

        COALESCE(pg.valor_multa_total, 0)
            AS valor_multa_total,

        COALESCE(pg.valor_juros_mora_total, 0)
            AS valor_juros_mora_total,

        COALESCE(pg.valor_desconto_total, 0)
            AS valor_desconto_total

    FROM parcelas_tratadas p

    LEFT JOIN pagamentos_por_parcela pg
        ON p.id_parcela = pg.id_parcela
),


-- ============================================================
-- 8. AGREGA PARCELAS
--    Grão: 1 linha por CONTRATO
-- ============================================================

metricas_por_contrato AS (

    SELECT
        id_contrato,

        COUNT(DISTINCT id_parcela)
            AS qtd_parcelas,

        MIN(data_vencimento)
            AS primeiro_vencimento,

        MAX(data_vencimento)
            AS ultimo_vencimento,

        SUM(COALESCE(valor_parcela, 0))
            AS valor_parcelas_total,

        SUM(COALESCE(valor_principal, 0))
            AS valor_principal_total,

        SUM(COALESCE(valor_juros, 0))
            AS valor_juros_total,

        CONCAT_WS(
            ' | ',
            SORT_ARRAY(
                COLLECT_SET(status_parcela)
            )
        ) AS status_parcelas,

        SUM(qtd_pagamentos)
            AS qtd_pagamentos,

        MIN(data_primeiro_pagamento)
            AS data_primeiro_pagamento,

        MAX(data_ultimo_pagamento)
            AS data_ultimo_pagamento,

        SUM(valor_pago_total)
            AS valor_pago_total,

        SUM(valor_multa_total)
            AS valor_multa_total,

        SUM(valor_juros_mora_total)
            AS valor_juros_mora_total,

        SUM(valor_desconto_total)
            AS valor_desconto_total

    FROM parcelas_enriquecidas

    GROUP BY
        id_contrato
),


-- ============================================================
-- 9. CONTRATOS + MÉTRICAS FINANCEIRAS
--    Grão: 1 linha por CONTRATO
-- ============================================================

contratos_enriquecidos AS (

    SELECT
        c.id_contrato,
        c.id_proposta,
        c.data_contratacao,
        c.valor_liberado,
        c.taxa_mensal_final,
        c.prazo_final_meses,
        c.status_contrato,
        c.canal_originacao,
        c.produto_credito_final,

        COALESCE(m.qtd_parcelas, 0)
            AS qtd_parcelas,

        m.primeiro_vencimento,
        m.ultimo_vencimento,

        COALESCE(m.valor_parcelas_total, 0)
            AS valor_parcelas_total,

        COALESCE(m.valor_principal_total, 0)
            AS valor_principal_total,

        COALESCE(m.valor_juros_total, 0)
            AS valor_juros_total,

        m.status_parcelas,

        COALESCE(m.qtd_pagamentos, 0)
            AS qtd_pagamentos,

        m.data_primeiro_pagamento,
        m.data_ultimo_pagamento,

        COALESCE(m.valor_pago_total, 0)
            AS valor_pago_total,

        COALESCE(m.valor_multa_total, 0)
            AS valor_multa_total,

        COALESCE(m.valor_juros_mora_total, 0)
            AS valor_juros_mora_total,

        COALESCE(m.valor_desconto_total, 0)
            AS valor_desconto_total

    FROM contratos_tratados c

    LEFT JOIN metricas_por_contrato m
        ON c.id_contrato = m.id_contrato
),


-- ============================================================
-- 10. AGREGA TUDO PARA O GRÃO DE PROPOSTA
-- ============================================================

metricas_por_proposta AS (

    SELECT
        id_proposta,

        COUNT(DISTINCT id_contrato)
            AS qtd_contratos,

        MIN(data_contratacao)
            AS data_primeira_contratacao,

        MAX(data_contratacao)
            AS data_ultima_contratacao,

        SUM(COALESCE(valor_liberado, 0))
            AS valor_liberado_total,

        AVG(taxa_mensal_final)
            AS taxa_mensal_final_media,

        AVG(prazo_final_meses)
            AS prazo_final_meses_medio,

        CONCAT_WS(
            ' | ',
            SORT_ARRAY(
                COLLECT_SET(status_contrato)
            )
        ) AS status_contratos,

        CONCAT_WS(
            ' | ',
            SORT_ARRAY(
                COLLECT_SET(canal_originacao)
            )
        ) AS canais_originacao,

        CONCAT_WS(
            ' | ',
            SORT_ARRAY(
                COLLECT_SET(produto_credito_final)
            )
        ) AS produtos_credito_finais,

        SUM(qtd_parcelas)
            AS qtd_parcelas,

        MIN(primeiro_vencimento)
            AS primeiro_vencimento,

        MAX(ultimo_vencimento)
            AS ultimo_vencimento,

        SUM(valor_parcelas_total)
            AS valor_parcelas_total,

        SUM(valor_principal_total)
            AS valor_principal_total,

        SUM(valor_juros_total)
            AS valor_juros_total,

        CONCAT_WS(
            ' | ',
            SORT_ARRAY(
                COLLECT_SET(status_parcelas)
            )
        ) AS status_parcelas,

        SUM(qtd_pagamentos)
            AS qtd_pagamentos,

        MIN(data_primeiro_pagamento)
            AS data_primeiro_pagamento,

        MAX(data_ultimo_pagamento)
            AS data_ultimo_pagamento,

        SUM(valor_pago_total)
            AS valor_pago_total,

        SUM(valor_multa_total)
            AS valor_multa_total,

        SUM(valor_juros_mora_total)
            AS valor_juros_mora_total,

        SUM(valor_desconto_total)
            AS valor_desconto_total

    FROM contratos_enriquecidos

    GROUP BY
        id_proposta
)


-- ============================================================
-- 11. ONE BIG TABLE FINAL
-- ============================================================

SELECT

    -- --------------------------------------------------------
    -- CLIENTE
    -- --------------------------------------------------------

    c.id_cliente,
    c.tipo_cliente,
    c.documento,
    c.data_cadastro,
    c.cidade,
    c.uf,
    c.segmento,
    c.score_credito,
    c.renda_mensal,


    -- --------------------------------------------------------
    -- PROPOSTA
    -- --------------------------------------------------------

    p.id_proposta,
    p.data_proposta,
    p.canal_origem,
    p.produto_credito,
    p.valor_solicitado,
    p.prazo_meses,
    p.taxa_mensal_solicitada,
    p.status_proposta,
    p.motivo_recusa,


    -- --------------------------------------------------------
    -- CONTRATOS
    -- --------------------------------------------------------

    COALESCE(m.qtd_contratos, 0)
        AS qtd_contratos,

    CASE
        WHEN COALESCE(m.qtd_contratos, 0) > 0
        THEN 1
        ELSE 0
    END AS flag_tem_contrato,

    m.data_primeira_contratacao,
    m.data_ultima_contratacao,

    COALESCE(m.valor_liberado_total, 0)
        AS valor_liberado_total,

    m.taxa_mensal_final_media,
    m.prazo_final_meses_medio,

    m.status_contratos,
    m.canais_originacao,
    m.produtos_credito_finais,


    -- --------------------------------------------------------
    -- PARCELAS
    -- --------------------------------------------------------

    COALESCE(m.qtd_parcelas, 0)
        AS qtd_parcelas,

    m.primeiro_vencimento,
    m.ultimo_vencimento,

    COALESCE(m.valor_parcelas_total, 0)
        AS valor_parcelas_total,

    COALESCE(m.valor_principal_total, 0)
        AS valor_principal_total,

    COALESCE(m.valor_juros_total, 0)
        AS valor_juros_total,

    m.status_parcelas,


    -- --------------------------------------------------------
    -- PAGAMENTOS
    -- --------------------------------------------------------

    COALESCE(m.qtd_pagamentos, 0)
        AS qtd_pagamentos,

    m.data_primeiro_pagamento,
    m.data_ultimo_pagamento,

    COALESCE(m.valor_pago_total, 0)
        AS valor_pago_total,

    COALESCE(m.valor_multa_total, 0)
        AS valor_multa_total,

    COALESCE(m.valor_juros_mora_total, 0)
        AS valor_juros_mora_total,

    COALESCE(m.valor_desconto_total, 0)
        AS valor_desconto_total,


    -- --------------------------------------------------------
    -- INDICADORES ANALÍTICOS PRONTOS
    -- --------------------------------------------------------

    CASE

        WHEN p.valor_solicitado > 0

        THEN ROUND(
            COALESCE(m.valor_liberado_total, 0)
            / p.valor_solicitado,
            4
        )

        ELSE NULL

    END AS indice_liberado_sobre_solicitado,


    CASE

        WHEN c.renda_mensal > 0

        THEN ROUND(
            COALESCE(m.valor_liberado_total, 0)
            / c.renda_mensal,
            4
        )

        ELSE NULL

    END AS indice_credito_sobre_renda,


    CASE

        WHEN m.data_primeira_contratacao IS NOT NULL

        THEN DATEDIFF(
            m.data_primeira_contratacao,
            p.data_proposta
        )

        ELSE NULL

    END AS dias_proposta_ate_contratacao,


    COALESCE(m.valor_multa_total, 0)
    +
    COALESCE(m.valor_juros_mora_total, 0)
        AS valor_mora_total


FROM propostas_tratadas p

LEFT JOIN clientes_tratados c
    ON p.id_cliente = c.id_cliente

LEFT JOIN metricas_por_proposta m
    ON p.id_proposta = m.id_proposta

""")


# ============================================================
# VISUALIZAÇÃO / VALIDAÇÃO RÁPIDA
# ============================================================

print("One Big Table criada com sucesso!")

print(
    "Quantidade de linhas:",
    resultado_final.count()
)

print(
    "Quantidade de propostas distintas:",
    resultado_final.select("id_proposta").distinct().count()
)

resultado_final.show(
    20,
    truncate=False
)

One Big Table criada com sucesso!
Quantidade de linhas: 910
Quantidade de propostas distintas: 901
+----------+------------+--------------+-------------+--------------+---+-----------+-------------+------------+-----------+-------------+-------------+---------------+----------------+-----------+----------------------+---------------+------------------+-------------+-----------------+-------------------------+-----------------------+--------------------+-----------------------+-----------------------+----------------+-----------------+-----------------------+------------+-------------------+-----------------+--------------------+---------------------+-----------------+------------------------+--------------+-----------------------+---------------------+----------------+-----------------+----------------------+--------------------+--------------------------------+--------------------------+-----------------------------+----------------+
|id_cliente|tipo_cliente|documento     |data_cadast

In [4]:
# ============================================================
# EXPORTAR ONE BIG TABLE PARA CSV
# ============================================================

from google.colab import files

# Converter Spark DataFrame -> Pandas
resultado_pandas = resultado_final.toPandas()

# Exportar CSV
resultado_pandas.to_csv(
    "resultado_final.csv",
    index=False,
    encoding="utf-8-sig"
)

print("resultado_final.csv gerado com sucesso!")

# Download
files.download("resultado_final.csv")

resultado_final.csv gerado com sucesso!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>